# Disconnected Grid Benchmark

Ноутбук для paper-style benchmark по мотивам Rao et al. на synthetic `2x2` grids. Он сравнивает `IG`, `NAA` и grid-конфигурации `Cheap-IG` на `Oxford Pets` в трёх setting'ах: `GridPG`, `DiPart`, `DiFull`, а также на двух evaluation-уровнях: `input` и `model.6`.

В benchmark входят quantitative localization score, outside/impossible positive mass, pairwise win-rates, `AggAtt` и Gaussian smoothing attribution-карт.

In [1]:
from pathlib import Path

from IPython.display import Markdown, display

from modules.disconnected_grid_benchmark import (
    benchmark_classifier_disconnected_grid,
    classifier_method_spec,
    render_disconnected_grid_report,
)


In [2]:
OXFORD_PETS_DIR = Path("oxford_pets")
N_IMAGES = 100

SPLIT_LAYER_NAME = "model.6"
EVALUATION_LAYERS = ["input", "model.6"]
SETTINGS = ["gridpg", "dipart", "difull"]
GRID_SIZE = 2
GRID_IMAGE_SIZE = 224
N_STEPS = 128
PREVIEW_IMAGES = 5
CLEAR_EVERY = 8
FD_EPS = 1e-3

INPUT_SMOOTHING_KERNELS = [33, 65, 129]
LAYER_SMOOTHING_KERNELS = [5, 9, 17]

CHEAP_IG_SEGMENT_START = 0.0
CHEAP_IG_SEGMENT_END = 0.2
CHEAP_IG_SELECTION_MODE = "positive"
CHEAP_IG_SELECTION_TOP_K_VALUES = [8000, 16000, 32000]

CACHE_ROOT = Path("output/disconnected_grid_cache")
OUTPUT_DIR = Path("output/disconnected_grid_classifier_oxford_pets_100")
REFRESH_CORE = False
REFRESH_METHODS = False
REFRESH_EVALUATIONS = False


In [3]:
def collect_image_paths(root: Path, n_images: int):
    exts = {".jpg", ".jpeg", ".png", ".webp"}
    paths = sorted(
        [path for path in root.iterdir() if path.suffix.lower() in exts],
        key=lambda path: path.name.lower(),
    )
    return [str(path) for path in paths[:n_images]]


IMAGE_PATHS = collect_image_paths(OXFORD_PETS_DIR, N_IMAGES)
len(IMAGE_PATHS), IMAGE_PATHS[:3]


(100,
 ['oxford_pets/Abyssinian_1.jpg',
  'oxford_pets/Abyssinian_108.jpg',
  'oxford_pets/Abyssinian_117.jpg'])

In [4]:
def build_cheap_ig_variants(fill_mode, fill_rho=None):
    variants = []
    for top_k in CHEAP_IG_SELECTION_TOP_K_VALUES:
        suffix = fill_mode if fill_mode == "zero" else f"{fill_mode}/rho{fill_rho:g}"
        variants.append(
            classifier_method_spec(
                "cheap_ig",
                name=f"Cheap-IG+[0,0.2]/k{top_k}/{suffix}",
                segment_start=CHEAP_IG_SEGMENT_START,
                segment_end=CHEAP_IG_SEGMENT_END,
                selection_mode=CHEAP_IG_SELECTION_MODE,
                selection_top_k=top_k,
                fill_mode=fill_mode,
                fill_rho=fill_rho if fill_rho is not None else 0.8,
            )
        )
    return variants


METHOD_SPECS = [
    classifier_method_spec("ig", name="IG"),
    classifier_method_spec("naa", name="NAA"),
    *build_cheap_ig_variants("zero"),
    *build_cheap_ig_variants("naa_scaled", fill_rho=0.8),
    *build_cheap_ig_variants("naa_scaled", fill_rho=1.0),
]

len(METHOD_SPECS), [spec["name"] for spec in METHOD_SPECS]


(11,
 ['IG',
  'NAA',
  'Cheap-IG+[0,0.2]/k8000/zero',
  'Cheap-IG+[0,0.2]/k16000/zero',
  'Cheap-IG+[0,0.2]/k32000/zero',
  'Cheap-IG+[0,0.2]/k8000/naa_scaled/rho0.8',
  'Cheap-IG+[0,0.2]/k16000/naa_scaled/rho0.8',
  'Cheap-IG+[0,0.2]/k32000/naa_scaled/rho0.8',
  'Cheap-IG+[0,0.2]/k8000/naa_scaled/rho1',
  'Cheap-IG+[0,0.2]/k16000/naa_scaled/rho1',
  'Cheap-IG+[0,0.2]/k32000/naa_scaled/rho1'])

In [5]:
results = benchmark_classifier_disconnected_grid(
    image_paths=IMAGE_PATHS,
    method_specs=METHOD_SPECS,
    split_layer_name=SPLIT_LAYER_NAME,
    evaluation_layers=EVALUATION_LAYERS,
    settings=SETTINGS,
    grid_size=GRID_SIZE,
    grid_image_size=GRID_IMAGE_SIZE,
    n_steps=N_STEPS,
    input_smoothing_kernels=INPUT_SMOOTHING_KERNELS,
    layer_smoothing_kernels=LAYER_SMOOTHING_KERNELS,
    preview_images=PREVIEW_IMAGES,
    clear_every=CLEAR_EVERY,
    fd_eps=FD_EPS,
    cache_root=CACHE_ROOT,
    target_dir=OUTPUT_DIR,
    save_output=True,
    refresh_core=REFRESH_CORE,
    refresh_methods=REFRESH_METHODS,
    refresh_evaluations=REFRESH_EVALUATIONS,
    verbose=False,
)

print("output_dir:", results["output_dir"])
print("report_path:", results["report_path"])
print("summary_path:", results["summary_path"])


output_dir: /Users/ashentide/PycharmProjects/PaperImplementations/output/disconnected_grid_classifier_oxford_pets_100
report_path: /Users/ashentide/PycharmProjects/PaperImplementations/output/disconnected_grid_classifier_oxford_pets_100/disconnected_grid_report.md
summary_path: /Users/ashentide/PycharmProjects/PaperImplementations/output/disconnected_grid_classifier_oxford_pets_100/disconnected_grid_summary.json


In [6]:
artifacts = render_disconnected_grid_report(results, output_dir=OUTPUT_DIR)
artifacts["report_path"], artifacts["summary_path"]


('output/disconnected_grid_classifier_oxford_pets_100/disconnected_grid_report.md',
 'output/disconnected_grid_classifier_oxford_pets_100/disconnected_grid_summary.json')

In [7]:
display(Markdown(Path(artifacts["report_path"]).read_text(encoding="utf-8")))


# Disconnected Grid Benchmark

Classifier-only paper-style benchmark on synthetic `2x2` grids for `IG`, `NAA` and `Cheap-IG`.

## Configuration

- split_layer_name=`model.6`
- evaluation_layers=`['input', 'model.6']`
- settings=`['gridpg', 'dipart', 'difull']`
- grid_size=`2`
- grid_image_size=`224`
- n_steps=`128`
- n_images=`100`
- cache_root=`output/disconnected_grid_cache`
- smoothing_kernels=`{'input': [33, 65, 129], 'model.6': [5, 9, 17]}`

## Best Variant Summary

| Setting | Layer | Method | Raw mean | Best variant | Best mean | Outside mean | runtime_s |
| --- | --- | --- | ---: | --- | ---: | ---: | ---: |
| difull | input | Cheap-IG+[0,0.2]/k16000/naa_scaled/rho0.8 | 1.0000 | raw | 1.0000 | 0.0000 | 4.2319 |
| difull | input | Cheap-IG+[0,0.2]/k16000/naa_scaled/rho1 | 1.0000 | raw | 1.0000 | 0.0000 | 4.2076 |
| difull | input | Cheap-IG+[0,0.2]/k16000/zero | 1.0000 | raw | 1.0000 | 0.0000 | 4.2330 |
| difull | input | Cheap-IG+[0,0.2]/k32000/naa_scaled/rho0.8 | 1.0000 | raw | 1.0000 | 0.0000 | 4.2125 |
| difull | input | Cheap-IG+[0,0.2]/k32000/naa_scaled/rho1 | 1.0000 | raw | 1.0000 | 0.0000 | 4.1863 |
| difull | input | Cheap-IG+[0,0.2]/k32000/zero | 1.0000 | raw | 1.0000 | 0.0000 | 4.2276 |
| difull | input | Cheap-IG+[0,0.2]/k8000/naa_scaled/rho0.8 | 1.0000 | raw | 1.0000 | 0.0000 | 4.2531 |
| difull | input | Cheap-IG+[0,0.2]/k8000/naa_scaled/rho1 | 1.0000 | raw | 1.0000 | 0.0000 | 4.2281 |
| difull | input | Cheap-IG+[0,0.2]/k8000/zero | 1.0000 | raw | 1.0000 | 0.0000 | 4.2330 |
| difull | input | IG | 1.0000 | raw | 1.0000 | 0.0000 | 6.2653 |
| difull | input | NAA | 1.0000 | raw | 1.0000 | 0.0000 | 3.5485 |
| difull | model.6 | Cheap-IG+[0,0.2]/k16000/naa_scaled/rho0.8 | 1.0000 | raw | 1.0000 | 0.0000 | 3.9848 |
| difull | model.6 | Cheap-IG+[0,0.2]/k16000/naa_scaled/rho1 | 1.0000 | raw | 1.0000 | 0.0000 | 3.9826 |
| difull | model.6 | Cheap-IG+[0,0.2]/k16000/zero | 1.0000 | raw | 1.0000 | 0.0000 | 3.9092 |
| difull | model.6 | Cheap-IG+[0,0.2]/k32000/naa_scaled/rho0.8 | 1.0000 | raw | 1.0000 | 0.0000 | 3.9431 |
| difull | model.6 | Cheap-IG+[0,0.2]/k32000/naa_scaled/rho1 | 1.0000 | raw | 1.0000 | 0.0000 | 3.9626 |
| difull | model.6 | Cheap-IG+[0,0.2]/k32000/zero | 1.0000 | raw | 1.0000 | 0.0000 | 3.9095 |
| difull | model.6 | Cheap-IG+[0,0.2]/k8000/naa_scaled/rho0.8 | 1.0000 | raw | 1.0000 | 0.0000 | 3.9354 |
| difull | model.6 | Cheap-IG+[0,0.2]/k8000/naa_scaled/rho1 | 1.0000 | raw | 1.0000 | 0.0000 | 3.9577 |
| difull | model.6 | Cheap-IG+[0,0.2]/k8000/zero | 1.0000 | raw | 1.0000 | 0.0000 | 3.9086 |
| difull | model.6 | IG | 1.0000 | raw | 1.0000 | 0.0000 | 5.9278 |
| difull | model.6 | NAA | 1.0000 | raw | 1.0000 | 0.0000 | 3.2613 |
| dipart | input | Cheap-IG+[0,0.2]/k16000/naa_scaled/rho0.8 | 0.6487 | raw | 0.6487 | 860.8606 | 6.7499 |
| dipart | input | Cheap-IG+[0,0.2]/k16000/naa_scaled/rho1 | 0.6425 | raw | 0.6425 | 889.8875 | 6.7110 |
| dipart | input | Cheap-IG+[0,0.2]/k16000/zero | 0.6749 | raw | 0.6749 | 745.9109 | 6.8698 |
| dipart | input | Cheap-IG+[0,0.2]/k32000/naa_scaled/rho0.8 | 0.6099 | raw | 0.6099 | 1059.1421 | 6.7385 |
| dipart | input | Cheap-IG+[0,0.2]/k32000/naa_scaled/rho1 | 0.6081 | raw | 0.6081 | 1068.0955 | 6.7498 |
| dipart | input | Cheap-IG+[0,0.2]/k32000/zero | 0.6169 | raw | 0.6169 | 1023.8948 | 6.8242 |
| dipart | input | Cheap-IG+[0,0.2]/k8000/naa_scaled/rho0.8 | 0.6900 | raw | 0.6900 | 660.1606 | 6.7457 |
| dipart | input | Cheap-IG+[0,0.2]/k8000/naa_scaled/rho1 | 0.6782 | raw | 0.6782 | 708.7357 | 6.7351 |
| dipart | input | Cheap-IG+[0,0.2]/k8000/zero | 0.7454 | raw | 0.7454 | 466.9741 | 6.8105 |
| dipart | input | IG | 0.5731 | gaussian_k129 | 0.8792 | 1.4870 | 12.0674 |
| dipart | input | NAA | 0.5730 | gaussian_k129 | 0.8802 | 1.4561 | 5.3086 |
| dipart | model.6 | Cheap-IG+[0,0.2]/k16000/naa_scaled/rho0.8 | 0.7229 | gaussian_k17 | 0.7254 | 7826.7830 | 5.7639 |
| dipart | model.6 | Cheap-IG+[0,0.2]/k16000/naa_scaled/rho1 | 0.7219 | gaussian_k17 | 0.7243 | 7876.0841 | 5.6851 |
| dipart | model.6 | Cheap-IG+[0,0.2]/k16000/zero | 0.7268 | gaussian_k17 | 0.7296 | 7631.6270 | 5.6913 |
| dipart | model.6 | Cheap-IG+[0,0.2]/k32000/naa_scaled/rho0.8 | 0.7260 | gaussian_k17 | 0.7298 | 7596.8222 | 5.7450 |
| dipart | model.6 | Cheap-IG+[0,0.2]/k32000/naa_scaled/rho1 | 0.7260 | gaussian_k17 | 0.7298 | 7596.8222 | 5.7836 |
| dipart | model.6 | Cheap-IG+[0,0.2]/k32000/zero | 0.7260 | gaussian_k17 | 0.7298 | 7596.8222 | 5.7666 |
| dipart | model.6 | Cheap-IG+[0,0.2]/k8000/naa_scaled/rho0.8 | 0.7276 | gaussian_k17 | 0.7289 | 7633.3335 | 5.7419 |
| dipart | model.6 | Cheap-IG+[0,0.2]/k8000/naa_scaled/rho1 | 0.7246 | gaussian_k17 | 0.7258 | 7790.7017 | 5.7009 |
| dipart | model.6 | Cheap-IG+[0,0.2]/k8000/zero | 0.7396 | gaussian_k17 | 0.7418 | 7016.9881 | 5.7034 |
| dipart | model.6 | IG | 0.8550 | gaussian_k17 | 0.8698 | 473.4734 | 11.1099 |
| dipart | model.6 | NAA | 0.8271 | gaussian_k17 | 0.8462 | 388.1941 | 4.2821 |
| gridpg | input | Cheap-IG+[0,0.2]/k16000/naa_scaled/rho0.8 | 0.3189 | raw | 0.3189 | 904.9968 | 4.3846 |
| gridpg | input | Cheap-IG+[0,0.2]/k16000/naa_scaled/rho1 | 0.3178 | raw | 0.3178 | 923.0727 | 4.3457 |
| gridpg | input | Cheap-IG+[0,0.2]/k16000/zero | 0.3234 | gaussian_k65 | 0.3240 | 756.5669 | 4.3851 |
| gridpg | input | Cheap-IG+[0,0.2]/k32000/naa_scaled/rho0.8 | 0.3132 | gaussian_k65 | 0.3141 | 941.0622 | 4.4754 |
| gridpg | input | Cheap-IG+[0,0.2]/k32000/naa_scaled/rho1 | 0.3129 | gaussian_k65 | 0.3138 | 947.5596 | 4.4295 |
| gridpg | input | Cheap-IG+[0,0.2]/k32000/zero | 0.3142 | gaussian_k65 | 0.3154 | 915.0766 | 4.3803 |
| gridpg | input | Cheap-IG+[0,0.2]/k8000/naa_scaled/rho0.8 | 0.3254 | raw | 0.3254 | 723.9193 | 4.3440 |
| gridpg | input | Cheap-IG+[0,0.2]/k8000/naa_scaled/rho1 | 0.3232 | raw | 0.3232 | 753.8216 | 4.4596 |
| gridpg | input | Cheap-IG+[0,0.2]/k8000/zero | 0.3357 | gaussian_k65 | 0.3361 | 563.5278 | 4.3860 |
| gridpg | input | IG | 0.2925 | gaussian_k129 | 0.5325 | 4.6661 | 6.3787 |
| gridpg | input | NAA | 0.2925 | gaussian_k129 | 0.5310 | 4.6065 | 3.6281 |
| gridpg | model.6 | Cheap-IG+[0,0.2]/k16000/naa_scaled/rho0.8 | 0.3453 | gaussian_k17 | 0.3466 | 15577.3329 | 4.0435 |
| gridpg | model.6 | Cheap-IG+[0,0.2]/k16000/naa_scaled/rho1 | 0.3451 | gaussian_k17 | 0.3464 | 15620.3987 | 3.9862 |
| gridpg | model.6 | Cheap-IG+[0,0.2]/k16000/zero | 0.3460 | gaussian_k17 | 0.3474 | 15406.8965 | 4.1013 |
| gridpg | model.6 | Cheap-IG+[0,0.2]/k32000/naa_scaled/rho0.8 | 0.3469 | gaussian_k17 | 0.3486 | 15616.7844 | 3.9422 |
| gridpg | model.6 | Cheap-IG+[0,0.2]/k32000/naa_scaled/rho1 | 0.3469 | gaussian_k17 | 0.3486 | 15616.7844 | 3.9683 |
| gridpg | model.6 | Cheap-IG+[0,0.2]/k32000/zero | 0.3469 | gaussian_k17 | 0.3486 | 15616.7844 | 4.1078 |
| gridpg | model.6 | Cheap-IG+[0,0.2]/k8000/naa_scaled/rho0.8 | 0.3443 | gaussian_k17 | 0.3452 | 15202.5563 | 4.0852 |
| gridpg | model.6 | Cheap-IG+[0,0.2]/k8000/naa_scaled/rho1 | 0.3437 | gaussian_k17 | 0.3445 | 15338.1672 | 3.9914 |
| gridpg | model.6 | Cheap-IG+[0,0.2]/k8000/zero | 0.3468 | gaussian_k17 | 0.3478 | 14669.9823 | 4.1047 |
| gridpg | model.6 | IG | 0.4486 | gaussian_k17 | 0.4676 | 1767.6574 | 6.2572 |
| gridpg | model.6 | NAA | 0.4362 | gaussian_k17 | 0.4643 | 750.8527 | 3.3884 |

## gridpg / input

![](output/disconnected_grid_classifier_oxford_pets_100/figures/summary_gridpg_input.png)

![](output/disconnected_grid_classifier_oxford_pets_100/figures/distribution_gridpg_input.png)

![](output/disconnected_grid_classifier_oxford_pets_100/figures/pairwise_gridpg_input.png)

![](output/disconnected_grid_classifier_oxford_pets_100/figures/aggatt_raw_gridpg_input.png)

![](output/disconnected_grid_classifier_oxford_pets_100/figures/aggatt_best_gridpg_input.png)

## gridpg / model.6

![](output/disconnected_grid_classifier_oxford_pets_100/figures/summary_gridpg_model_6.png)

![](output/disconnected_grid_classifier_oxford_pets_100/figures/distribution_gridpg_model_6.png)

![](output/disconnected_grid_classifier_oxford_pets_100/figures/pairwise_gridpg_model_6.png)

![](output/disconnected_grid_classifier_oxford_pets_100/figures/aggatt_raw_gridpg_model_6.png)

![](output/disconnected_grid_classifier_oxford_pets_100/figures/aggatt_best_gridpg_model_6.png)

## dipart / input

![](output/disconnected_grid_classifier_oxford_pets_100/figures/summary_dipart_input.png)

![](output/disconnected_grid_classifier_oxford_pets_100/figures/distribution_dipart_input.png)

![](output/disconnected_grid_classifier_oxford_pets_100/figures/pairwise_dipart_input.png)

![](output/disconnected_grid_classifier_oxford_pets_100/figures/aggatt_raw_dipart_input.png)

![](output/disconnected_grid_classifier_oxford_pets_100/figures/aggatt_best_dipart_input.png)

## dipart / model.6

![](output/disconnected_grid_classifier_oxford_pets_100/figures/summary_dipart_model_6.png)

![](output/disconnected_grid_classifier_oxford_pets_100/figures/distribution_dipart_model_6.png)

![](output/disconnected_grid_classifier_oxford_pets_100/figures/pairwise_dipart_model_6.png)

![](output/disconnected_grid_classifier_oxford_pets_100/figures/aggatt_raw_dipart_model_6.png)

![](output/disconnected_grid_classifier_oxford_pets_100/figures/aggatt_best_dipart_model_6.png)

## difull / input

![](output/disconnected_grid_classifier_oxford_pets_100/figures/summary_difull_input.png)

![](output/disconnected_grid_classifier_oxford_pets_100/figures/distribution_difull_input.png)

![](output/disconnected_grid_classifier_oxford_pets_100/figures/pairwise_difull_input.png)

![](output/disconnected_grid_classifier_oxford_pets_100/figures/aggatt_raw_difull_input.png)

![](output/disconnected_grid_classifier_oxford_pets_100/figures/aggatt_best_difull_input.png)

## difull / model.6

![](output/disconnected_grid_classifier_oxford_pets_100/figures/summary_difull_model_6.png)

![](output/disconnected_grid_classifier_oxford_pets_100/figures/distribution_difull_model_6.png)

![](output/disconnected_grid_classifier_oxford_pets_100/figures/pairwise_difull_model_6.png)

![](output/disconnected_grid_classifier_oxford_pets_100/figures/aggatt_raw_difull_model_6.png)

![](output/disconnected_grid_classifier_oxford_pets_100/figures/aggatt_best_difull_model_6.png)
